# Getting started with MemsArrayWS objects

The `MemsArrayWS` class allows getting signals from a remote antenna running a local *Megamicros Broadcast Server (MBS)* server. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros.core.ws import MemsArrayWS

log.setLevel( "INFO" )

# Set server access credentials
#HOST = 'buzenval20.fr'
#HOST = 'parisparc.biimea.tech
HOST = 'localhost'
PORT = 9002

## Connecting to the remote server

Providing a *MBS* server is running at ``HOST:PORT``, one can try to connect by creating a ``MemsArrayWS`` object.

In [2]:
# Define the antenna
try:
    antenna = MemsArrayWS( HOST, port=PORT )
except Exception as e:
    print( f"Failed: {e}" )


2023-11-04 00:45:01,549 [INFO]:  .Install MemsArrayWS settings
2023-11-04 00:45:01,550 [INFO]:  .Created a new antenna
2023-11-04 00:45:01,551 [INFO]:  .Async event loop already running. Adding coroutine to the event loop...


2023-11-04 00:45:01,556 [INFO]:  .Try connecting to ws://localhost:9002...
2023-11-04 00:45:01,584 [INFO]:  .Received positive answer from server
2023-11-04 00:45:01,584 [INFO]:  .Getting settings values from remote receiver...
2023-11-04 00:45:01,586 [INFO]:  .Received settings from server [ok]
2023-11-04 00:45:01,586 [INFO]:  .Set 8 available MEMs numbered from 0 to 7
2023-11-04 00:45:01,587 [INFO]:  .No analogic channels available
2023-11-04 00:45:01,589 [INFO]:  .Starting MegamicrosWS device [ready]


## Performing a selftest

In [ ]:
# Perform an antenna selftest
antenna.selftest()

## Halting the remote server
Notice that the conection is lost. As such you could not restart the server.

In [ ]:
# Stop the remote server
antenna.shutdown()


## Running

### Getting signals from some MEMs

In [ ]:
# 2 seconds run, getting signals from MEMs 1 and 2
antenna.run(
    mems = [1, 2],
    duration=2,
    buffer_length=512,
    signal_q_size = 0,
)

# Init a np.ndarray
signals = np.ndarray( (0, antenna.channels_number ) )

# Get signals
i = 0
for data in antenna:
    i += 1
    signals = np.concatenate( ( signals, data ), axis=0 )

# waiting for the end of the running thread is mandatory
antenna.wait()
print( f"Exit from loop. Received {i} frames. Signal shape is: {np.shape( signals )}" )

## Plotting signals

In [ ]:
# plot signals
time = np.array( range( np.size(signals,0) ) )/antenna.sampling_frequency
fig, axs = plt.subplots( antenna.channels_number )
fig.suptitle('Channels activity')	
for s in range( antenna.channels_number ):
    axs[s].plot( time, signals[:,s] )
    axs[s].set( xlabel='time in seconds', ylabel='channel %d' % s )

plt.show()

## Saving signals as wav file
Since wavfiles are audio files, you cannot save more than 2 channels.

In [ ]:
import wave

WAV_FILENAME = 'toto.wav'

# 2 seconds run, getting signals from MEMs 1 and 2
antenna.run(
    mems = [1, 2],
    duration=10,
    buffer_length=512,
    signal_q_size = 0,
)

with  wave.open( WAV_FILENAME, mode='wb' ) as wavfile:
    wavfile.setnchannels(2)
    wavfile.setsampwidth(2)
    wavfile.setframerate( antenna.sampling_frequency )

    # Get signals
    for data in antenna:
        signal = data >> 4
        wavfile.writeframesraw( np.int16( np.reshape( signal, np.size( signal ), order='F' ) ) )

# waiting for the end of the running thread is mandatory
antenna.wait()

## Hearing signal with *pyaudio* library

Note that `signal_q_size` is set to 0, setting the internal queue to an infinite length.
This prevents breaks in the audio stream.

In [6]:
import pyaudio

FRAME_LENGTH = 512
SAMPLING_FREQUENCY = 50000
antenna.setSamplingFrequency( SAMPLING_FREQUENCY )

# Instantiate PyAudio and initialize PortAudio system resources (1)
p = pyaudio.PyAudio()

# Open stream
stream = p.open(
    format = pyaudio.paFloat32,
    channels = 2,
    rate = int( antenna.sampling_frequency ),
    output=True,
    frames_per_buffer=FRAME_LENGTH,
)

# Start running the remote Megamicros system
antenna.run( 
    mems=[3, 4],
    duration=10,
    frame_length=FRAME_LENGTH,
    counter_skip = True,
    signal_q_size = 0
)

# Get signals
transfers_counter = 0
for data in antenna:
    signal = data >> 4

    # convert into float and normalize with MEMs sensibility
    data = ( data.astype( np.float32 ).T * antenna.sensibility )

    # write into audio stream
    stream.write( data, num_frames=FRAME_LENGTH )
    transfers_counter += 1

# Close stream and release PortAudio system resources (5)
stream.close()            
p.terminate()

antenna.wait()


2023-11-04 00:56:49,084 [INFO]:  .Starting run execution
2023-11-04 00:56:49,086 [INFO]:  .Install MemsArray settings
2023-11-04 00:56:49,088 [INFO]:  .2 MEMs were activated among 0 to 7 available MEMs
2023-11-04 00:56:49,090 [INFO]:  .Install MemsArrayWS settings
2023-11-04 00:56:49,091 [INFO]:  .Pre-execution checks for MemsArray.run()
2023-11-04 00:56:49,093 [INFO]:  .Requested job: run
2023-11-04 00:56:49,093 [INFO]:  .Perform a 10s run loop
2023-11-04 00:56:49,095 [INFO]:  .H5 recording off
2023-11-04 00:56:49,096 [INFO]:  .Start a `run` running job on remote server
2023-11-04 00:56:49,096 [INFO]:  .Background execution mode off
2023-11-04 00:56:49,097 [INFO]:  .Run thread execution started
2023-11-04 00:56:49,097 [INFO]:  .Starting iterations: will produce data as numpy array of int32 (512 x 2 size)
2023-11-04 00:56:49,100 [INFO]:  .Connecting to remote host localhost:9002...
2023-11-04 00:56:49,105 [INFO]:  .Connected
2023-11-04 00:56:49,106 [INFO]:  .Send running job command (r

## Saving signals as H5 files

You can save signal in H5 file format. In this example sigansl are saved on the MBS remote server.
The antenna receive no more signals. 

In [ ]:
antenna.run(
    mems = [3, 4],
    duration=2,
    buffer_length=512,
    h5_recording=True,                          # H5 recording ON
    h5_pass_through=True,                       # perform F5 recording on server
    h5_rootdir='./',                            # directory where to save file
    h5_compressing=False,                       # Use compression or not
    background_mode=True,
    signal_q_size = 0,
)

antenna.wait()

## Getting signals yourself

In this example, signals are received using the antenna internal queue.

In [ ]:
import queue

antenna.run(
    mems = [1, 2],
    duration=2,
    buffer_length=512,
    signal_q_size = 0,
)

i = 0
while True:
    try:
        data = antenna.signal_q.get( timeout=5 )
        print( f"[{i}]" )
        i += 1
        # do what you want with data...

    except queue.Empty:
        print( f"exit from loop at i={i}" )
        break

antenna.wait()

## Listening to the Megamicro remote server
By starting a *master* run on the server, you can connect to the server from others hosts and listening to the signal stream.

### Staring the master run
This call lets the remote server starting a run in the background mode.

In [ ]:

antenna.run(
    mems = [1, 2, 3, 4],
    duration=0,
    buffer_length=512,
    signal_q_size=0,
    job='master', 
)

In [ ]:
# Define the antenna
try:
    listener = MemsArrayWS( HOST, port=PORT )
except Exception as e:
    print( f"Failed: {e}" )


In [ ]:
listener.setFrameLength(512)
listener.run(
    mems = [1, 2],
    buffer_length=512,
    signal_q_size=0,
    duration=10,
    job='listen'
)

# Init a np.ndarray
signals = np.ndarray( (0, listener.channels_number ) )

# Get signals
i = 0
for data in listener:
    i += 1
    signals = np.concatenate( ( signals, data ), axis=0 )
    print( data )

# waiting for the end of the running thread is mandatory
listener.wait()
print( f"Exit from loop. Received {i} frames. Signal shape is: {np.shape( signals )}" )

In [ ]:
# halt job
antenna.halt()

In [ ]:
# halt server
antenna.shutdown()